# GPTCloneBench V3 clean-data function embeddings

This notebook embeds only the current **GPTCloneBench V3 clean-data release**.
It reads the attached Kaggle Dataset directly from `clean_data`, validates the
no-self-pair split, and generates CodeBERT, GraphCodeBERT, UniXcoder, and CodeT5
endpoint embeddings. The output Dataset also carries the exact pair table,
code metadata, and source metadata required by the downstream classification
notebook; the classifier therefore needs no separate benchmark input.


## 1. Kaggle runtime requirements

1. Enable Internet and a T4-class GPU.
2. Add `koushamoeini/gpt-clone-bench` as the only data input.
3. Select **Run All**; no path, filename, secret, or compression edit is needed.
4. When the run finishes, use **Save Version** so its `/kaggle/working`
   artifacts can be attached to the classification notebook as Notebook Output.

The loader supports both the original ZIP layout and Kaggle's nested temporary
mount such as `codes.jsonl/codes.jsonl.gz.tmp`. Dataset publication is
intentionally disabled: the saved notebook output is the only hand-off to the
next notebook, so a Kaggle API token and Dataset-status polling are unnecessary.


In [ ]:
import importlib.metadata
import subprocess
import sys

PACKAGES = [
    "transformers==4.57.6",
    "huggingface_hub>=0.34,<1.0",
    "accelerate>=1.0",
    "sentencepiece",
    "safetensors",
    "pyarrow>=20",
    "duckdb>=1.3",
    "tqdm",
    "Pillow>=10,<12",
    "kaggle==2.2.3",
    "kagglesdk==0.1.31",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        *PACKAGES,
    ]
)


def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not-installed"


print("Dependencies are ready.")

for package in [
    "torch",
    "transformers",
    "huggingface_hub",
    "accelerate",
    "sentencepiece",
    "safetensors",
    "pyarrow",
    "duckdb",
    "tqdm",
    "Pillow",
    "kaggle",
    "kagglesdk",
]:
    print(f"{package}: {package_version(package)}")

## 2. GPTCloneBench-only configuration

Only GPTCloneBench is processed. The output is a standalone Kaggle Dataset that
can be attached directly to `gptclonebench-classification.ipynb`.


In [ ]:
from pathlib import Path

PREFERRED_INPUT_DATASET_ROOT = Path("/kaggle/input/datasets/koushamoeini/gpt-clone-bench/clean_data")
INPUT_SEARCH_ROOT = Path("/kaggle/input")
INPUT_DATASET_HANDLE = "koushamoeini/gpt-clone-bench"
EXPECTED_SOURCE_DATA_FORMAT = "spectral_clean_data_v1"
EXPECTED_SOURCE_DATASET_KEY = "gptclonebench_v3"
EXPECTED_SOURCE_CONTENT_VERSION = EXPECTED_SOURCE_DATA_FORMAT

KAGGLE_OWNER = "kousha.m"
OUTPUT_DATASET_SLUG = "gptclonebench-v3-clean-function-embeddings"
OUTPUT_DATASET_TITLE = "GPTCloneBench V3 Clean Function Embeddings"
OUTPUT_DATASET_SUBTITLE = "No-self-pair GPTCloneBench embeddings and exact split artifacts"
EMBEDDING_PIPELINE_VERSION = "3.0.0-gptclonebench-clean"
OUTPUT_VERSION_NOTES = (
    "GPTCloneBench-only embeddings for the no-self-pair V3 clean-data release"
)

DATASETS_TO_RUN = ["GPTCloneBench"]
MODELS_TO_RUN = ["CodeBERT", "GraphCodeBERT", "UniXCode", "CodeT5"]

MAX_SEQUENCE_LENGTH = 512
WINDOW_OVERLAP_TOKENS = 64
FUNCTION_BATCH_SIZE = 64
ROWS_PER_SHARD = 25_000
VALIDATION_BATCH_SIZE = 2_048
MODEL_WINDOW_BATCH_SIZES = {
    "CodeBERT": 32,
    "GraphCodeBERT": 24,
    "UniXCode": 24,
    "CodeT5": 16,
}

INFERENCE_DTYPE = "float16"
OUTPUT_DTYPE = "auto"
L2_NORMALIZE = False
MAX_SAFE_OUTPUT_GIB = 18.0
OUTPUT_ESTIMATE_SAFETY_FACTOR = 1.20

QUICK_TEST = False
QUICK_TEST_ROWS_PER_DATASET = 200
RESUME_FROM_EXISTING_OUTPUT = False
ALLOW_VERSION_IF_EXISTS = True
REQUIRE_EXISTING_OUTPUT_DATASET = False
FORCE_REBUILD = True
OUTPUT_DATASET_VISIBILITY = "private"
PUBLISH_TO_KAGGLE = False
PUBLISH_AFTER_EACH_MODEL = False
REQUIRE_CUDA = True

WORK_ROOT = Path("/kaggle/temp/gptclonebench_embedding_builder_v3_clean")
SOURCE_DOWNLOAD_ROOT = WORK_ROOT / "source_archive"
PREPARED_ROOT = WORK_ROOT / "prepared_functions"
RESUME_DOWNLOAD_ROOT = WORK_ROOT / "existing_output"
HF_CACHE_ROOT = WORK_ROOT / "hf_cache"
DUCKDB_TEMP_ROOT = WORK_ROOT / "duckdb_temp"
OUTPUT_ROOT = Path("/kaggle/working/gptclonebench-v3-clean-function-embeddings")

for path_value in [
    WORK_ROOT, SOURCE_DOWNLOAD_ROOT, PREPARED_ROOT, RESUME_DOWNLOAD_ROOT,
    HF_CACHE_ROOT, DUCKDB_TEMP_ROOT, OUTPUT_ROOT,
]:
    path_value.mkdir(parents=True, exist_ok=True)

print("Embedding pipeline version:", EMBEDDING_PIPELINE_VERSION)
print("Preferred input directory:", PREFERRED_INPUT_DATASET_ROOT)
print("Fallback search root:", INPUT_SEARCH_ROOT)
print("Output directory:", OUTPUT_ROOT)


## 3. Imports, environment, and reproducibility

In [ ]:
from __future__ import annotations

import csv
import gc
import io
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterator, Optional

# Keep model and source caches outside /kaggle/working so they are not counted
# as Notebook output artifacts.
os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_ROOT / "transformers")
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import duckdb
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from huggingface_hub import model_info
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoTokenizer, T5EncoderModel

SEED = 20260701
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if REQUIRE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError(
        "CUDA is unavailable. Enable a Kaggle GPU accelerator and restart."
    )

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(properties.total_memory / 2**30, 2), "GiB")

## 4. Kaggle authentication and CLI helpers

Credentials are read only from Kaggle Secrets or pre-existing environment
variables. They are never printed or written into the output Dataset.

In [ ]:
def get_kaggle_secret(name: str) -> Optional[str]:
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(name)
        return value.strip() if value else None
    except Exception:
        value = os.environ.get(name)
        return value.strip() if value else None


def configure_kaggle_auth(required: bool = True) -> str:
    username = KAGGLE_OWNER
    api_token = get_kaggle_secret("KAGGLE_API_TOKEN")
    legacy_key = get_kaggle_secret("KAGGLE_KEY")

    if not username:
        if required:
            raise RuntimeError("KAGGLE_USERNAME is missing.")
        return "missing"

    os.environ["KAGGLE_USERNAME"] = username

    if api_token:
        os.environ["KAGGLE_API_TOKEN"] = api_token
        return "api_token"

    if legacy_key:
        os.environ["KAGGLE_KEY"] = legacy_key
        return "legacy_key"

    if required:
        raise RuntimeError("KAGGLE_API_TOKEN is missing.")

    return "missing"


def run_command(
    command: list[str],
    *,
    check: bool = True,
    quiet: bool = False,
    timeout: int = 600,
) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(
        command,
        text=True,
        capture_output=True,
        timeout=timeout,
        env=os.environ.copy(),
    )

    if not quiet and result.stdout.strip():
        print(result.stdout.strip())

    if check and result.returncode != 0:
        combined = "\n".join(
            part.strip()
            for part in (result.stdout or "", result.stderr or "")
            if part and part.strip()
        )
        raise RuntimeError(
            "Command failed:\n" + " ".join(command) + "\n\n" + combined
        )

    return result


def combined_output(result: subprocess.CompletedProcess[str]) -> str:
    return "\n".join(
        part.strip()
        for part in (result.stdout or "", result.stderr or "")
        if part and part.strip()
    )


def owned_dataset_exists(handle: str) -> bool:
    owner, slug = handle.split("/", 1)

    if owner.casefold() != KAGGLE_USERNAME.casefold():
        raise RuntimeError(
            "The Dataset handle owner does not match KAGGLE_USERNAME."
        )

    result = run_command(
        [
            "kaggle",
            "datasets",
            "list",
            "--mine",
            "--search",
            slug,
            "--page",
            "1",
            "--csv",
        ],
        check=True,
        quiet=True,
        timeout=180,
    )

    rows = list(csv.reader(io.StringIO((result.stdout or "").strip())))
    if not rows:
        return False

    header = [column.strip().casefold() for column in rows[0]]
    ref_index = header.index("ref") if "ref" in header else 0

    refs = {
        row[ref_index].strip().casefold()
        for row in rows[1:]
        if len(row) > ref_index and "/" in row[ref_index]
    }
    return handle.casefold() in refs


AUTH_METHOD = configure_kaggle_auth(required=False)
KAGGLE_USERNAME = KAGGLE_OWNER
OUTPUT_DATASET_HANDLE = f"{KAGGLE_USERNAME}/{OUTPUT_DATASET_SLUG}"

# Publication is disabled. Optional credentials remain available only for the
# one-time public-source download fallback if an attached mount is incomplete.
EARLY_OUTPUT_DATASET_EXISTS = False
if PUBLISH_TO_KAGGLE:
    if AUTH_METHOD == "missing":
        raise RuntimeError("KAGGLE_API_TOKEN is required only when publication is enabled.")
    run_command(
        ["kaggle", "datasets", "list", "--mine", "--page", "1"],
        quiet=True,
        timeout=180,
    )
    EARLY_OUTPUT_DATASET_EXISTS = owned_dataset_exists(OUTPUT_DATASET_HANDLE)
    if REQUIRE_EXISTING_OUTPUT_DATASET and not EARLY_OUTPUT_DATASET_EXISTS:
        raise RuntimeError(f"Required output Dataset not found: {OUTPUT_DATASET_HANDLE}")

print("Kaggle authentication:", AUTH_METHOD)
print("Publication mode:", "Dataset" if PUBLISH_TO_KAGGLE else "saved_notebook_output")


## 5. Discover and validate the GPTCloneBench clean-data input

Only source code and pair labels are required. The loader supports both the
original ZIP layout (`clean_data/codes.jsonl.gz` and `pairs.csv.gz`) and
Kaggle's mounted layout, where a large compressed JSONL may appear as
`clean_data/codes.jsonl/codes.jsonl.gz.tmp`. Compression is detected from
the gzip magic bytes. Graph spectra are optional. Counts, endpoint coverage,
self-pairs, language labels, and split isolation are validated before loading
any transformer checkpoint.


In [ ]:
import pandas as pd

EXPECTED_FUNCTION_ROWS = 5_924
EXPECTED_V3_PAIR_COUNTS = {
    "GPTCloneBench": {"train": 4_144, "valid": 886, "test": 894}
}
EXPECTED_V3_POSITIVE_COUNTS = {
    "GPTCloneBench": {"train": 2_072, "valid": 443, "test": 447}
}
CODE_FILE_NAMES = ("codes.jsonl", "codes.jsonl.gz", "codes.jsonl.gz.tmp")
PAIR_FILE_NAMES = ("pairs.csv", "pairs.csv.gz", "pairs.csv.gz.tmp")
METADATA_FILE_NAMES = ("metadata.json",)
GRAPH_FILE_NAMES = (
    "graph_spectra.jsonl", "graph_spectra.jsonl.gz", "graph_spectra.jsonl.gz.tmp"
)
CODE_CONTAINER_NAMES = ("codes.jsonl", "codes.jsonl.gz")
PAIR_CONTAINER_NAMES = ("pairs.csv", "pairs.csv.gz")
GRAPH_CONTAINER_NAMES = ("graph_spectra.jsonl", "graph_spectra.jsonl.gz")

def read_json_file(path: Path) -> dict[str, Any]:
    value = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise RuntimeError(f"Expected a JSON object in {path}.")
    return value

def first_role_file(
    root: Path,
    names: tuple[str, ...],
    container_names: tuple[str, ...] = (),
) -> Optional[Path]:
    # Normal extracted-ZIP layout.
    direct = next((root / name for name in names if (root / name).is_file()), None)
    if direct is not None:
        return direct

    # Kaggle can expose a large source file as a directory and place the actual
    # gzip stream in a *.gz.tmp child, e.g. codes.jsonl/codes.jsonl.gz.tmp.
    for container_name in container_names:
        container = root / container_name
        if not container.is_dir():
            continue
        for name in names:
            nested = container / name
            if nested.is_file():
                return nested
        nested_files = sorted(path for path in container.rglob("*") if path.is_file())
        for nested in nested_files:
            lowered = nested.name.casefold()
            if any(lowered == name.casefold() for name in names):
                return nested
    return None

def detected_pandas_compression(path: Path) -> Optional[str]:
    # The Kaggle-mounted *.tmp file still contains gzip bytes.
    with path.open("rb") as stream:
        magic = stream.read(2)
    return "gzip" if magic == b"\x1f\x8b" else None

def clean_root_files(root: Path) -> Optional[dict[str, Optional[Path]]]:
    if not root.is_dir():
        return None
    codes_path = first_role_file(root, CODE_FILE_NAMES, CODE_CONTAINER_NAMES)
    pairs_path = first_role_file(root, PAIR_FILE_NAMES, PAIR_CONTAINER_NAMES)
    metadata_path = first_role_file(root, METADATA_FILE_NAMES)
    if codes_path is None or pairs_path is None or metadata_path is None:
        return None
    return {
        "codes": codes_path,
        "pairs": pairs_path,
        "metadata": metadata_path,
        "graph_spectra": first_role_file(
            root, GRAPH_FILE_NAMES, GRAPH_CONTAINER_NAMES
        ),
    }

def candidate_clean_roots(base: Path) -> list[Path]:
    candidates = [PREFERRED_INPUT_DATASET_ROOT, base]
    if base.is_dir():
        for name in (*CODE_FILE_NAMES, *PAIR_FILE_NAMES, *METADATA_FILE_NAMES):
            candidates.extend(path.parent for path in base.rglob(name))
    unique, seen = [], set()
    for candidate in candidates:
        key = candidate.resolve().as_posix() if candidate.exists() else candidate.as_posix()
        if key not in seen:
            seen.add(key)
            unique.append(candidate)
    return unique

def find_clean_root(base: Path) -> tuple[Optional[Path], Optional[dict[str, Optional[Path]]]]:
    for candidate in candidate_clean_roots(base):
        files = clean_root_files(candidate)
        if files is not None:
            return candidate, files
    return None, None

SOURCE_ROOT, SOURCE_FILES = find_clean_root(INPUT_SEARCH_ROOT)
SOURCE_INPUT_METHOD = "attached_kaggle_input"

if SOURCE_ROOT is None:
    download_root = SOURCE_DOWNLOAD_ROOT / "kaggle_api_download"
    shutil.rmtree(download_root, ignore_errors=True)
    download_root.mkdir(parents=True, exist_ok=True)
    print("Attached mount has no complete code/pair bundle; downloading once:", INPUT_DATASET_HANDLE)
    run_command([
        "kaggle", "datasets", "download", "-d", INPUT_DATASET_HANDLE,
        "--path", str(download_root), "--unzip", "--force",
    ], timeout=7_200)
    SOURCE_ROOT, SOURCE_FILES = find_clean_root(download_root)
    SOURCE_INPUT_METHOD = "kaggle_api_download"

if SOURCE_ROOT is None or SOURCE_FILES is None:
    visible = []
    if INPUT_SEARCH_ROOT.is_dir():
        for path in INPUT_SEARCH_ROOT.rglob("*"):
            if path.is_file() and any(token in path.name.casefold() for token in ("code", "pair", "metadata", "spectra")):
                visible.append(str(path))
                if len(visible) >= 100:
                    break
    raise FileNotFoundError(
        "Could not locate a complete GPTCloneBench input containing metadata.json, "
        "codes.jsonl[.gz], and pairs.csv[.gz]. Visible related files:\n" +
        "\n".join(visible)
    )

SOURCE_METADATA_PATH = SOURCE_FILES["metadata"]
PAIRS_PATH = SOURCE_FILES["pairs"]
CODES_PATH = SOURCE_FILES["codes"]
GRAPH_SPECTRA_PATH = SOURCE_FILES["graph_spectra"]
SOURCE_METADATA = read_json_file(SOURCE_METADATA_PATH)
SOURCE_VERSION_INDEX = SOURCE_METADATA

if str(SOURCE_METADATA.get("format")) != EXPECTED_SOURCE_DATA_FORMAT:
    raise RuntimeError(f"Unexpected clean-data format: {SOURCE_METADATA.get('format')!r}")
if str(SOURCE_METADATA.get("dataset_key")) != EXPECTED_SOURCE_DATASET_KEY:
    raise RuntimeError(f"Unexpected dataset_key: {SOURCE_METADATA.get('dataset_key')!r}")

source_pairs = pd.read_csv(
    PAIRS_PATH, compression=detected_pandas_compression(PAIRS_PATH)
)
required_pair_columns = {"split", "left_id", "right_id", "label"}
missing_pair_columns = sorted(required_pair_columns - set(source_pairs.columns))
if missing_pair_columns:
    raise RuntimeError(f"Pair file is missing columns: {missing_pair_columns}")

source_pairs["split"] = source_pairs["split"].astype(str).str.casefold().replace({"validation": "valid"})
source_pairs["label"] = source_pairs["label"].astype(int)
source_pairs["left_id"] = source_pairs["left_id"].astype(str)
source_pairs["right_id"] = source_pairs["right_id"].astype(str)

observed_rows = source_pairs.groupby("split", observed=True).size().astype(int).to_dict()
observed_positive = source_pairs.loc[source_pairs["label"].eq(1)].groupby("split", observed=True).size().astype(int).to_dict()
if observed_rows != EXPECTED_V3_PAIR_COUNTS["GPTCloneBench"]:
    raise RuntimeError(f"Pair counts changed: {observed_rows}")
if observed_positive != EXPECTED_V3_POSITIVE_COUNTS["GPTCloneBench"]:
    raise RuntimeError(f"Positive counts changed: {observed_positive}")
if set(source_pairs["label"].unique()) != {0, 1}:
    raise RuntimeError("Labels must be exactly {0, 1}.")
if source_pairs["left_id"].eq(source_pairs["right_id"]).any():
    raise RuntimeError("The clean GPTCloneBench release unexpectedly contains self-pairs.")

source_codes = pd.read_json(
    CODES_PATH, lines=True, compression=detected_pandas_compression(CODES_PATH)
)
required_code_columns = {"code_id", "code", "language"}
missing_code_columns = sorted(required_code_columns - set(source_codes.columns))
if missing_code_columns:
    raise RuntimeError(f"Code file is missing columns: {missing_code_columns}")
source_codes["code_id"] = source_codes["code_id"].astype(str)
source_codes["code"] = source_codes["code"].astype(str)
source_codes["language"] = source_codes["language"].astype(str).str.casefold()
if len(source_codes) != EXPECTED_FUNCTION_ROWS or source_codes["code_id"].nunique() != EXPECTED_FUNCTION_ROWS:
    raise RuntimeError("The clean release must contain exactly 5,924 unique code rows.")
if source_codes["code"].str.strip().eq("").any():
    raise RuntimeError("At least one source function is empty.")
if set(source_codes["language"].unique()) != {"c", "csharp", "java", "python"}:
    raise RuntimeError("Unexpected GPTCloneBench language set.")

code_ids = set(source_codes["code_id"])
endpoint_ids = set(source_pairs["left_id"]) | set(source_pairs["right_id"])
if endpoint_ids != code_ids:
    raise RuntimeError("Pair endpoints do not exactly match the 5,924 code IDs.")
endpoint_split_rows = pd.concat([
    source_pairs[["split", "left_id"]].rename(columns={"left_id": "code_id"}),
    source_pairs[["split", "right_id"]].rename(columns={"right_id": "code_id"}),
], ignore_index=True)
endpoint_cross_split = int(endpoint_split_rows.groupby("code_id")["split"].nunique().gt(1).sum())
if endpoint_cross_split:
    raise RuntimeError(f"Endpoint leakage across splits: {endpoint_cross_split}")

print("Source root:", SOURCE_ROOT)
print("Input method:", SOURCE_INPUT_METHOD)
print("Codes file:", CODES_PATH.name)
print("Pairs file:", PAIRS_PATH.name)
print("Graph spectra:", GRAPH_SPECTRA_PATH.name if GRAPH_SPECTRA_PATH else "not required / not present")
print("Functions:", f"{len(source_codes):,}")
print("Pairs:", observed_rows)
print("Self-pairs: 0")
print("Endpoint cross-split leakage: 0")


## 6. Prepare the GPTCloneBench endpoint source

The 5,924 rows from `codes.jsonl` are converted to a temporary two-column
Parquet source with `function_id` and `code`. The exact `pairs.csv`,
`codes.jsonl`, and source metadata are copied into the embedding output for
the classification notebook.


In [ ]:
def sql_path(path: Path | str) -> str:
    return str(path).replace("\\", "/").replace("'", "''")

def new_duckdb_connection() -> duckdb.DuckDBPyConnection:
    connection = duckdb.connect()
    connection.execute("PRAGMA threads=4")
    connection.execute("PRAGMA memory_limit='12GB'")
    connection.execute(f"PRAGMA temp_directory='{sql_path(DUCKDB_TEMP_ROOT)}'")
    connection.execute("PRAGMA preserve_insertion_order=false")
    return connection

DATASETS_TO_RUN = ["GPTCloneBench"]
V3_SPLIT_NAMES = ("train", "valid", "test")
V3_PROTOCOL_RELATIVE_DIRS = {"GPTCloneBench": "pairs.csv"}
SELECTION_DESCRIPTIONS = {
    "GPTCloneBench": "all 5,924 endpoints in the no-self-pair GPTCloneBench V3 clean-data release"
}

prepared_source_path = PREPARED_ROOT / "gptclonebench.parquet"
PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
prepared_frame = (
    source_codes[["code_id", "code"]]
    .rename(columns={"code_id": "function_id"})
    .sort_values("function_id")
    .reset_index(drop=True)
)
if QUICK_TEST:
    prepared_frame = prepared_frame.head(QUICK_TEST_ROWS_PER_DATASET).copy()
prepared_frame.to_parquet(prepared_source_path, index=False, compression="zstd")

FULL_SOURCE_COUNTS = {"GPTCloneBench": EXPECTED_FUNCTION_ROWS}
SOURCE_COUNTS = {"GPTCloneBench": len(prepared_frame)}
FULL_SOURCE_QUALITY = {
    "GPTCloneBench": {
        "rows": EXPECTED_FUNCTION_ROWS,
        "unique_ids": EXPECTED_FUNCTION_ROWS,
        "null_ids": 0,
        "empty_code_rows": 0,
    }
}
FUNCTION_FILES = {"GPTCloneBench": prepared_source_path}

selection_manifest = {
    "embedding_pipeline_version": EMBEDDING_PIPELINE_VERSION,
    "source_dataset_root": str(SOURCE_ROOT),
    "source_input_method": SOURCE_INPUT_METHOD,
    "source_content_version": EXPECTED_SOURCE_CONTENT_VERSION,
    "source_data_format": EXPECTED_SOURCE_DATA_FORMAT,
    "source_dataset_key": EXPECTED_SOURCE_DATASET_KEY,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "selection_scope": "complete_no_self_pair_release",
    "selected_function_counts": SOURCE_COUNTS,
    "full_selected_function_counts": FULL_SOURCE_COUNTS,
    "source_quality": FULL_SOURCE_QUALITY,
    "selection_descriptions": SELECTION_DESCRIPTIONS,
    "version_3_pair_counts": EXPECTED_V3_PAIR_COUNTS,
    "version_3_positive_counts": EXPECTED_V3_POSITIVE_COUNTS,
    "self_pair_rows": 0,
    "endpoint_cross_split_violations": 0,
    "quick_test": QUICK_TEST,
}

def copy_source_artifacts() -> None:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    source_pairs.to_csv(OUTPUT_ROOT / "pairs.csv", index=False)
    source_codes.to_json(OUTPUT_ROOT / "codes.jsonl", orient="records", lines=True, force_ascii=False)
    shutil.copy2(SOURCE_METADATA_PATH, OUTPUT_ROOT / "source_metadata.json")
    (OUTPUT_ROOT / "source_selection_manifest.json").write_text(
        json.dumps(selection_manifest, indent=2, sort_keys=True), encoding="utf-8"
    )

copy_source_artifacts()
print("Prepared function rows:", f"{SOURCE_COUNTS['GPTCloneBench']:,}")


## 7. Preflight the expected GPTCloneBench output size

The estimate covers four 768-dimensional vectors for each selected function.


In [ ]:
MODEL_DIMENSION = 768
total_functions_per_model = sum(SOURCE_COUNTS.values())


def estimate_output_bytes(dtype_name: str) -> tuple[int, int]:
    bytes_per_value = {"float16": 2, "float32": 4}[dtype_name]
    raw_bytes = (
        total_functions_per_model
        * len(MODELS_TO_RUN)
        * MODEL_DIMENSION
        * bytes_per_value
    )
    conservative_bytes = int(raw_bytes * OUTPUT_ESTIMATE_SAFETY_FACTOR)
    return raw_bytes, conservative_bytes


float32_raw_bytes, float32_estimated_bytes = estimate_output_bytes("float32")
float16_raw_bytes, float16_estimated_bytes = estimate_output_bytes("float16")
safe_limit_bytes = int(MAX_SAFE_OUTPUT_GIB * 2**30)

if OUTPUT_DTYPE == "auto":
    if float32_estimated_bytes <= safe_limit_bytes:
        OUTPUT_DTYPE = "float32"
    elif float16_estimated_bytes <= safe_limit_bytes:
        OUTPUT_DTYPE = "float16"
    else:
        raise RuntimeError(
            "Even float16 embeddings exceed the configured safe Kaggle "
            "Notebook output threshold. Reduce MODELS_TO_RUN or split the "
            "work across separate output Datasets."
        )
elif OUTPUT_DTYPE not in {"float16", "float32"}:
    raise ValueError(
        "OUTPUT_DTYPE must be 'auto', 'float16', or 'float32'."
    )

raw_vector_bytes, conservative_estimated_bytes = estimate_output_bytes(
    OUTPUT_DTYPE
)

print("Total selected functions per model:", f"{total_functions_per_model:,}")
print("Models:", len(MODELS_TO_RUN))
print("Selected output dtype:", OUTPUT_DTYPE)
print("Float32 conservative estimate:", f"{float32_estimated_bytes / 2**30:.2f} GiB")
print("Float16 conservative estimate:", f"{float16_estimated_bytes / 2**30:.2f} GiB")
print("Selected raw vector payload:", f"{raw_vector_bytes / 2**30:.2f} GiB")
print(
    "Selected conservative output estimate:",
    f"{conservative_estimated_bytes / 2**30:.2f} GiB",
)
print("Configured safe limit:", f"{MAX_SAFE_OUTPUT_GIB:.2f} GiB")

if conservative_estimated_bytes > safe_limit_bytes:
    raise RuntimeError(
        "The conservative embedding output estimate exceeds the safe Kaggle "
        "Notebook output threshold. Run a subset of MODELS_TO_RUN and publish "
        "additional versions or separate Datasets."
    )

SIZE_PREFLIGHT = {
    "selected_functions_per_model": total_functions_per_model,
    "models": len(MODELS_TO_RUN),
    "dimension": MODEL_DIMENSION,
    "selected_output_dtype": OUTPUT_DTYPE,
    "float32_raw_gib": float32_raw_bytes / 2**30,
    "float32_conservative_gib": float32_estimated_bytes / 2**30,
    "float16_raw_gib": float16_raw_bytes / 2**30,
    "float16_conservative_gib": float16_estimated_bytes / 2**30,
    "selected_raw_gib": raw_vector_bytes / 2**30,
    "selected_conservative_gib": conservative_estimated_bytes / 2**30,
    "safe_limit_gib": MAX_SAFE_OUTPUT_GIB,
}


In [ ]:
def prepared_path(dataset_name: str) -> Path:
    if dataset_name != "GPTCloneBench":
        raise KeyError(dataset_name)
    return FUNCTION_FILES[dataset_name]

if not prepared_source_path.is_file():
    raise FileNotFoundError(prepared_source_path)
if len(prepared_frame) != SOURCE_COUNTS["GPTCloneBench"]:
    raise RuntimeError("Prepared GPTCloneBench function count changed.")

copy_source_artifacts()


## 8. Embedding methodology

### Pooling

- **CodeBERT:** first `<s>`/CLS hidden state.
- **GraphCodeBERT:** first `<s>`/CLS hidden state.
- **UniXcoder:** official encoder-only masked mean.
- **CodeT5:** masked mean over encoder hidden states; T5 has no CLS token.

### Long functions

Code is tokenized without truncating its tail and divided into overlapping
windows of at most 512 model tokens. Window vectors are aggregated by a
newly-covered-token weighted mean so overlap tokens are not over-counted.

### GraphCodeBERT input mode

GPTCloneBench contains C, C#, Java, and Python. The released downstream DFG
preprocessing does not provide one uniform path for all four languages, so the
GraphCodeBERT checkpoint is used in sequence-only mode. This limitation is
stored in each shard and in the output manifest.


In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    name: str
    model_id: str
    architecture: str
    pooling: str
    input_mode: str
    expected_dimension: int = 768
    max_length: int = 512
    language_scope: str = ""


MODEL_SPECS: dict[str, ModelSpec] = {
    "CodeBERT": ModelSpec(
        name="CodeBERT",
        model_id="microsoft/codebert-base",
        architecture="roberta",
        pooling="cls",
        input_mode="raw_code_sequence",
        language_scope=(
            "Pre-trained on Java, Python, JavaScript, PHP, Ruby, and Go; "
            "C and C# rows are evaluated out of the reported pre-training scope."
        ),
    ),
    "GraphCodeBERT": ModelSpec(
        name="GraphCodeBERT",
        model_id="microsoft/graphcodebert-base",
        architecture="roberta",
        pooling="cls",
        input_mode="raw_code_sequence_without_explicit_dfg",
        language_scope=(
            "Sequence-only checkpoint inference is used consistently for all "
            "languages; no explicit data-flow graph is constructed."
        ),
    ),
    "UniXCode": ModelSpec(
        name="UniXCode",
        model_id="microsoft/unixcoder-base-nine",
        architecture="unixcoder",
        pooling="masked_mean",
        input_mode="encoder-only",
        language_scope=(
            "The nine-language checkpoint includes Java, Python, C, and C#."
        ),
    ),
    "CodeT5": ModelSpec(
        name="CodeT5",
        model_id="Salesforce/codet5-base",
        architecture="t5_encoder",
        pooling="masked_mean",
        input_mode="raw_code_sequence",
        language_scope="Multilingual CodeT5 base checkpoint.",
    ),
}

unknown_models = sorted(set(MODELS_TO_RUN) - set(MODEL_SPECS))
if unknown_models:
    raise ValueError(f"Unknown model names: {unknown_models}")

for model_name in MODELS_TO_RUN:
    print(json.dumps(asdict(MODEL_SPECS[model_name]), indent=2))

## 9. Load checkpoints and construct overlapping model windows

In [ ]:
def resolve_model_revision(model_id: str) -> str:
    return model_info(model_id).sha


def load_embedding_model(spec: ModelSpec):
    revision = resolve_model_revision(spec.model_id)

    tokenizer = AutoTokenizer.from_pretrained(
        spec.model_id,
        revision=revision,
        use_fast=(spec.architecture != "unixcoder"),
        cache_dir=HF_CACHE_ROOT,
    )

    model_kwargs: dict[str, Any] = {
        "revision": revision,
        "cache_dir": HF_CACHE_ROOT,
        "low_cpu_mem_usage": True,
    }

    if DEVICE.type == "cuda":
        if INFERENCE_DTYPE == "float16":
            model_kwargs["torch_dtype"] = torch.float16
        elif INFERENCE_DTYPE == "float32":
            model_kwargs["torch_dtype"] = torch.float32
        else:
            raise ValueError(
                f"Unsupported INFERENCE_DTYPE: {INFERENCE_DTYPE}"
            )

    if spec.architecture == "t5_encoder":
        model = T5EncoderModel.from_pretrained(
            spec.model_id,
            **model_kwargs,
        )
    elif spec.architecture == "unixcoder":
        config = AutoConfig.from_pretrained(
            spec.model_id,
            revision=revision,
            cache_dir=HF_CACHE_ROOT,
        )
        config.is_decoder = True
        model = AutoModel.from_pretrained(
            spec.model_id,
            config=config,
            add_pooling_layer=False,
            **model_kwargs,
        )
    else:
        model = AutoModel.from_pretrained(
            spec.model_id,
            add_pooling_layer=False,
            **model_kwargs,
        )

    model.eval()
    model.to(DEVICE)

    mode_token_id = None
    if spec.architecture == "unixcoder":
        mode_token_id = tokenizer.convert_tokens_to_ids("<encoder-only>")
        if mode_token_id == tokenizer.unk_token_id:
            raise RuntimeError(
                "The UniXcoder tokenizer does not expose <encoder-only>."
            )

    hidden_size_value = getattr(model.config, "hidden_size", None)
    if hidden_size_value is None:
        hidden_size_value = getattr(model.config, "d_model")
    hidden_size = int(hidden_size_value)
    if hidden_size != spec.expected_dimension:
        raise RuntimeError(
            f"{spec.name} hidden size is {hidden_size}; expected "
            f"{spec.expected_dimension}."
        )

    return tokenizer, model, revision, mode_token_id


def content_capacity(spec: ModelSpec, tokenizer) -> int:
    special_tokens = (
        4
        if spec.architecture == "unixcoder"
        else tokenizer.num_special_tokens_to_add(pair=False)
    )
    capacity = min(MAX_SEQUENCE_LENGTH, spec.max_length) - special_tokens
    if capacity <= WINDOW_OVERLAP_TOKENS:
        raise ValueError("WINDOW_OVERLAP_TOKENS is too large.")
    return capacity


def build_model_sequence(
    chunk_ids: list[int],
    spec: ModelSpec,
    tokenizer,
    mode_token_id: Optional[int],
) -> list[int]:
    if spec.architecture == "unixcoder":
        return [
            tokenizer.cls_token_id,
            mode_token_id,
            tokenizer.sep_token_id,
            *chunk_ids,
            tokenizer.sep_token_id,
        ]
    return tokenizer.build_inputs_with_special_tokens(chunk_ids)


def iter_function_windows(
    token_ids: list[int],
    spec: ModelSpec,
    tokenizer,
    mode_token_id: Optional[int],
) -> Iterator[tuple[list[int], int]]:
    capacity = content_capacity(spec, tokenizer)
    step = capacity - WINDOW_OVERLAP_TOKENS

    if not token_ids:
        yield build_model_sequence([], spec, tokenizer, mode_token_id), 1
        return

    start = 0
    while start < len(token_ids):
        chunk = token_ids[start : start + capacity]
        sequence = build_model_sequence(
            chunk,
            spec,
            tokenizer,
            mode_token_id,
        )

        if len(sequence) > MAX_SEQUENCE_LENGTH:
            raise RuntimeError(
                f"Constructed sequence has {len(sequence)} tokens."
            )

        newly_covered_tokens = (
            len(chunk)
            if start == 0
            else max(1, len(chunk) - WINDOW_OVERLAP_TOKENS)
        )
        yield sequence, newly_covered_tokens

        if start + capacity >= len(token_ids):
            break
        start += step

## 10. Batched inference and function-level aggregation

In [ ]:
def infer_sequence_batch(
    sequences: list[list[int]],
    spec: ModelSpec,
    tokenizer,
    model,
) -> np.ndarray:
    if not sequences:
        return np.empty((0, spec.expected_dimension), dtype=np.float32)

    try:
        maximum_length = max(map(len, sequences))
        pad_token_id = tokenizer.pad_token_id
        if pad_token_id is None:
            raise RuntimeError(f"{spec.name} tokenizer has no pad token.")

        input_ids = torch.full(
            (len(sequences), maximum_length),
            fill_value=pad_token_id,
            dtype=torch.long,
            device=DEVICE,
        )
        valid_mask = torch.zeros(
            (len(sequences), maximum_length),
            dtype=torch.bool,
            device=DEVICE,
        )

        for row_index, sequence in enumerate(sequences):
            length = len(sequence)
            input_ids[row_index, :length] = torch.tensor(
                sequence,
                dtype=torch.long,
                device=DEVICE,
            )
            valid_mask[row_index, :length] = True

        if spec.architecture == "unixcoder":
            model_attention_mask = (
                valid_mask.unsqueeze(1) & valid_mask.unsqueeze(2)
            )
        else:
            model_attention_mask = valid_mask.long()

        with torch.inference_mode():
            autocast_enabled = (
                DEVICE.type == "cuda" and INFERENCE_DTYPE == "float16"
            )
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=autocast_enabled,
            ):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=model_attention_mask,
                )
                hidden = outputs.last_hidden_state

                if spec.pooling == "cls":
                    vectors = hidden[:, 0, :]
                elif spec.pooling == "masked_mean":
                    pool_mask = valid_mask.unsqueeze(-1).to(hidden.dtype)
                    vectors = (
                        (hidden * pool_mask).sum(dim=1)
                        / pool_mask.sum(dim=1).clamp_min(1)
                    )
                else:
                    raise ValueError(f"Unsupported pooling: {spec.pooling}")

                if L2_NORMALIZE:
                    vectors = F.normalize(vectors, p=2, dim=1)

        return vectors.float().cpu().numpy()

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(sequences) == 1:
            raise
        middle = len(sequences) // 2
        return np.concatenate(
            [
                infer_sequence_batch(
                    sequences[:middle], spec, tokenizer, model
                ),
                infer_sequence_batch(
                    sequences[middle:], spec, tokenizer, model
                ),
            ],
            axis=0,
        )


def embed_function_batch(
    codes: list[str],
    spec: ModelSpec,
    tokenizer,
    model,
    mode_token_id: Optional[int],
    window_batch_size: int,
) -> tuple[np.ndarray, list[int], list[int]]:
    tokenized = tokenizer(
        codes,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        verbose=False,
    )["input_ids"]

    sums = np.zeros(
        (len(codes), spec.expected_dimension),
        dtype=np.float64,
    )
    total_weights = np.zeros(len(codes), dtype=np.float64)
    token_counts = [len(ids) for ids in tokenized]
    window_counts = [0 for _ in codes]

    pending_sequences: list[list[int]] = []
    pending_owners: list[int] = []
    pending_weights: list[int] = []

    def flush_pending() -> None:
        nonlocal pending_sequences, pending_owners, pending_weights
        if not pending_sequences:
            return

        window_vectors = infer_sequence_batch(
            pending_sequences,
            spec,
            tokenizer,
            model,
        )
        for vector, owner, weight in zip(
            window_vectors,
            pending_owners,
            pending_weights,
        ):
            sums[owner] += vector.astype(np.float64) * weight
            total_weights[owner] += weight
            window_counts[owner] += 1

        pending_sequences = []
        pending_owners = []
        pending_weights = []

    for owner, token_ids in enumerate(tokenized):
        for sequence, weight in iter_function_windows(
            token_ids,
            spec,
            tokenizer,
            mode_token_id,
        ):
            pending_sequences.append(sequence)
            pending_owners.append(owner)
            pending_weights.append(weight)
            if len(pending_sequences) >= window_batch_size:
                flush_pending()

    flush_pending()

    if np.any(total_weights <= 0):
        raise RuntimeError("At least one function received no embedding window.")

    vectors = sums / total_weights[:, None]
    if L2_NORMALIZE:
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        vectors = vectors / np.maximum(norms, 1e-12)

    return vectors.astype(np.float32), token_counts, window_counts


def smoke_test_model(
    spec: ModelSpec,
    tokenizer,
    model,
    mode_token_id: Optional[int],
) -> dict[str, Any]:
    sample = "public int add(int a, int b) { return a + b; }"
    token_ids = tokenizer(
        sample,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]
    sequence, _ = next(
        iter_function_windows(token_ids, spec, tokenizer, mode_token_id)
    )

    if spec.pooling == "cls":
        if tokenizer.cls_token_id is None:
            raise RuntimeError(f"{spec.name} has no CLS token ID.")
        if sequence[0] != tokenizer.cls_token_id:
            raise RuntimeError(
                f"{spec.name} sequence does not begin with its CLS token."
            )

    vectors = infer_sequence_batch([sequence], spec, tokenizer, model)
    if vectors.shape != (1, spec.expected_dimension):
        raise RuntimeError(
            f"Unexpected smoke-test shape for {spec.name}: {vectors.shape}"
        )
    if not np.isfinite(vectors).all():
        raise RuntimeError(f"{spec.name} smoke-test vector is not finite.")
    norm = float(np.linalg.norm(vectors[0]))
    if norm <= 1e-12:
        raise RuntimeError(f"{spec.name} smoke-test vector has zero norm.")

    report = {
        "model": spec.name,
        "pooling": spec.pooling,
        "sequence_length": len(sequence),
        "dimension": spec.expected_dimension,
        "vector_norm": norm,
        "first_token_is_cls": (
            sequence[0] == tokenizer.cls_token_id
            if tokenizer.cls_token_id is not None
            else False
        ),
    }
    print(json.dumps(report, indent=2))
    return report

## 11. Parquet sharding, validation, and resume state

Files are flat at the Dataset root so the Kaggle CLI can upload them without
zipping directories. The model and benchmark are encoded in each filename.
The Parquet table itself contains only the requested three columns.

In [ ]:
def safe_slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.casefold()).strip("-")


def unit_prefix(model_name: str, dataset_name: str) -> str:
    return f"{safe_slug(model_name)}__{safe_slug(dataset_name)}"


def unit_shards(model_name: str, dataset_name: str) -> list[Path]:
    prefix = unit_prefix(model_name, dataset_name)
    return sorted(OUTPUT_ROOT.glob(f"{prefix}__part-*.parquet"))


def unit_success_path(model_name: str, dataset_name: str) -> Path:
    return OUTPUT_ROOT / f"{unit_prefix(model_name, dataset_name)}__SUCCESS.json"


def next_shard_index(model_name: str, dataset_name: str) -> int:
    pattern = re.compile(r"__part-(\d+)\.parquet$")
    indices = []
    for shard in unit_shards(model_name, dataset_name):
        match = pattern.search(shard.name)
        if match:
            indices.append(int(match.group(1)))
    return max(indices, default=-1) + 1


def completed_rows(model_name: str, dataset_name: str) -> int:
    return sum(
        pq.ParquetFile(shard).metadata.num_rows
        for shard in unit_shards(model_name, dataset_name)
    )


def decoded_parquet_metadata(path: Path) -> dict[str, str]:
    metadata = pq.ParquetFile(path).schema_arrow.metadata or {}
    return {
        key.decode("utf-8"): value.decode("utf-8")
        for key, value in metadata.items()
    }


def assert_existing_shards_compatible(
    model_name: str,
    dataset_name: str,
    expected_metadata: dict[str, str],
) -> None:
    keys_to_match = [
        "model_name",
        "model_id",
        "model_revision",
        "pooling",
        "input_mode",
        "max_sequence_length",
        "window_overlap_tokens",
        "window_aggregation",
        "l2_normalized",
        "inference_dtype",
        "output_dtype",
        "bigclonebench_selection",
        "embedding_pipeline_version",
        "source_content_version",
        "source_selection",
    ]

    mismatches: list[dict[str, Any]] = []
    for shard in unit_shards(model_name, dataset_name):
        actual = decoded_parquet_metadata(shard)
        differences = {
            key: {
                "expected": expected_metadata[key],
                "actual": actual.get(key),
            }
            for key in keys_to_match
            if actual.get(key) != expected_metadata[key]
        }
        if differences:
            mismatches.append(
                {"shard": shard.name, "differences": differences}
            )

    if mismatches:
        raise RuntimeError(
            "Existing shards are incompatible with the current model or "
            "embedding configuration. Set FORCE_REBUILD=True or use a new "
            "output Dataset slug. Mismatches: "
            + json.dumps(mismatches[:5], indent=2)
        )


def output_files_size() -> int:
    return sum(
        path.stat().st_size
        for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
    )


def enforce_output_size_limit() -> None:
    current_gib = output_files_size() / 2**30
    if current_gib > MAX_SAFE_OUTPUT_GIB:
        raise RuntimeError(
            f"Output reached {current_gib:.2f} GiB, above the configured "
            f"safe limit of {MAX_SAFE_OUTPUT_GIB:.2f} GiB."
        )


def embedding_arrow_type(dimension: int) -> pa.DataType:
    scalar_type = pa.float16() if OUTPUT_DTYPE == "float16" else pa.float32()
    return pa.list_(scalar_type, list_size=dimension)


def write_embedding_shard(
    model_name: str,
    dataset_name: str,
    shard_index: int,
    function_ids: list[str],
    vectors: np.ndarray,
    dimension: int,
    metadata: dict[str, str],
) -> Path:
    prefix = unit_prefix(model_name, dataset_name)
    final_path = OUTPUT_ROOT / f"{prefix}__part-{shard_index:05d}.parquet"
    temporary_path = final_path.with_suffix(".parquet.tmp")
    temporary_path.unlink(missing_ok=True)

    output_numpy_dtype = np.float16 if OUTPUT_DTYPE == "float16" else np.float32
    matrix = np.ascontiguousarray(vectors.astype(output_numpy_dtype, copy=False))

    scalar_type = pa.float16() if OUTPUT_DTYPE == "float16" else pa.float32()
    flat_values = pa.array(matrix.reshape(-1), type=scalar_type)
    vector_array = pa.FixedSizeListArray.from_arrays(
        flat_values,
        list_size=dimension,
    )

    schema = pa.schema(
        [
            pa.field("function_id", pa.string(), nullable=False),
            pa.field("dataset", pa.string(), nullable=False),
            pa.field(
                "embedding",
                embedding_arrow_type(dimension),
                nullable=False,
            ),
        ]
    ).with_metadata(
        {
            key.encode("utf-8"): value.encode("utf-8")
            for key, value in metadata.items()
        }
    )

    table = pa.Table.from_arrays(
        [
            pa.array(function_ids, type=pa.string()),
            pa.array([dataset_name] * len(function_ids), type=pa.string()),
            vector_array,
        ],
        schema=schema,
    )

    pq.write_table(
        table,
        temporary_path,
        compression="zstd",
        compression_level=6,
        use_dictionary=["function_id", "dataset"],
        write_statistics=True,
        row_group_size=min(len(function_ids), 10_000),
    )
    temporary_path.replace(final_path)
    enforce_output_size_limit()
    return final_path


def validate_unit(
    model_name: str,
    dataset_name: str,
    expected_rows: int,
    expected_dimension: int,
) -> dict[str, Any]:
    shards = unit_shards(model_name, dataset_name)
    if not shards:
        return {"valid": False, "reason": "no_shards", "rows": 0}

    expected_type = embedding_arrow_type(expected_dimension)
    total_rows = 0
    invalid_vectors = 0
    zero_norm_vectors = 0
    minimum_norm = math.inf
    maximum_norm = 0.0

    for shard in shards:
        parquet_file = pq.ParquetFile(shard)
        schema = parquet_file.schema_arrow
        if schema.names != ["function_id", "dataset", "embedding"]:
            raise ValueError(f"Unexpected columns in {shard}: {schema.names}")
        if schema.field("embedding").type != expected_type:
            raise ValueError(
                f"Unexpected embedding type in {shard}: "
                f"{schema.field('embedding').type}; expected {expected_type}"
            )

        for batch in parquet_file.iter_batches(
            batch_size=VALIDATION_BATCH_SIZE,
            columns=["embedding"],
        ):
            total_rows += batch.num_rows
            embeddings = batch.column(0)
            invalid_vectors += embeddings.null_count
            values = embeddings.values.to_numpy(
                zero_copy_only=False
            ).reshape(batch.num_rows, expected_dimension)
            finite = np.isfinite(values).all(axis=1)
            invalid_vectors += int((~finite).sum())
            norms = np.linalg.norm(
                values.astype(np.float32, copy=False), axis=1
            )
            zero_norm_vectors += int((norms <= 1e-12).sum())
            if len(norms):
                minimum_norm = min(minimum_norm, float(norms.min()))
                maximum_norm = max(maximum_norm, float(norms.max()))

    glob_path = sql_path(
        OUTPUT_ROOT / f"{unit_prefix(model_name, dataset_name)}__part-*.parquet"
    )
    connection = new_duckdb_connection()
    try:
        relational = connection.execute(
            f"""
            SELECT
                COUNT(*) AS rows,
                COUNT(DISTINCT function_id) AS unique_ids,
                SUM(CASE WHEN dataset <> ? THEN 1 ELSE 0 END) AS wrong_dataset,
                SUM(CASE WHEN function_id IS NULL THEN 1 ELSE 0 END) AS null_ids
            FROM read_parquet('{glob_path}')
            """,
            [dataset_name],
        ).fetchone()
    finally:
        connection.close()

    relational_rows, unique_ids, wrong_dataset, null_ids = map(int, relational)
    valid = (
        total_rows == expected_rows
        and relational_rows == expected_rows
        and unique_ids == expected_rows
        and wrong_dataset == 0
        and null_ids == 0
        and invalid_vectors == 0
        and zero_norm_vectors == 0
    )

    return {
        "valid": valid,
        "rows": total_rows,
        "expected_rows": expected_rows,
        "unique_function_ids": unique_ids,
        "wrong_dataset_rows": wrong_dataset,
        "null_function_ids": null_ids,
        "invalid_vectors": invalid_vectors,
        "zero_norm_vectors": zero_norm_vectors,
        "minimum_vector_norm": (
            None if math.isinf(minimum_norm) else minimum_norm
        ),
        "maximum_vector_norm": maximum_norm,
        "dimension": expected_dimension,
        "shards": len(shards),
    }

## 12. Start with a clean GPTCloneBench result snapshot

With `FORCE_REBUILD=True`, only the local Kaggle working output is cleared;
the attached clean-data archive is never modified. If the private output
Dataset already exists it is versioned, otherwise it is created privately.


In [ ]:
def copy_resume_files(source: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    allowed_patterns = [
        "*.parquet",
        "*__SUCCESS.json",
        "embedding_manifest.json",
        "source_selection_manifest.json",
        "final_validation.json",
    ]
    for pattern in allowed_patterns:
        for path in source.rglob(pattern):
            if path.is_file():
                shutil.copy2(path, destination / path.name)


if FORCE_REBUILD:
    # The source-selection manifest was created during source preparation.
    # Recreate it after clearing any stale local embedding output.
    shutil.rmtree(OUTPUT_ROOT, ignore_errors=True)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    copy_source_artifacts()

OUTPUT_DATASET_EXISTS = owned_dataset_exists(OUTPUT_DATASET_HANDLE)
print("Existing private result Dataset found:", OUTPUT_DATASET_EXISTS)

if REQUIRE_EXISTING_OUTPUT_DATASET and not OUTPUT_DATASET_EXISTS:
    raise RuntimeError(
        f"The result Dataset {OUTPUT_DATASET_HANDLE!r} does not exist. "
        "This notebook is configured to use the existing "
        "result Dataset, not a separate Dataset."
    )

if (
    OUTPUT_DATASET_EXISTS
    and RESUME_FROM_EXISTING_OUTPUT
    and not FORCE_REBUILD
    and not QUICK_TEST
):
    print("Downloading compatible GPTCloneBench embeddings for resume...")
    shutil.rmtree(RESUME_DOWNLOAD_ROOT, ignore_errors=True)
    RESUME_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

    run_command(
        [
            "kaggle",
            "datasets",
            "download",
            OUTPUT_DATASET_HANDLE,
            "--path",
            str(RESUME_DOWNLOAD_ROOT),
            "--unzip",
            "--force",
        ],
        timeout=7_200,
    )
    copy_resume_files(RESUME_DOWNLOAD_ROOT, OUTPUT_ROOT)
    print("Existing compatible embedding shards copied.")
elif OUTPUT_DATASET_EXISTS and not ALLOW_VERSION_IF_EXISTS:
    raise RuntimeError(
        f"Dataset {OUTPUT_DATASET_HANDLE!r} already exists and versioning "
        "is disabled."
    )
else:
    print("Starting a clean GPTCloneBench embedding snapshot.")


## 13. Generate one model–dataset unit

In [ ]:
def iter_source_batches(
    dataset_name: str,
    rows_to_skip: int,
) -> Iterator[tuple[list[str], list[str]]]:
    parquet_file = pq.ParquetFile(FUNCTION_FILES[dataset_name])
    remaining_skip = rows_to_skip

    for record_batch in parquet_file.iter_batches(
        batch_size=FUNCTION_BATCH_SIZE,
        columns=["function_id", "code"],
    ):
        if remaining_skip >= record_batch.num_rows:
            remaining_skip -= record_batch.num_rows
            continue
        if remaining_skip > 0:
            record_batch = record_batch.slice(remaining_skip)
            remaining_skip = 0

        function_ids = [str(value) for value in record_batch.column(0).to_pylist()]
        codes = [str(value) for value in record_batch.column(1).to_pylist()]
        if function_ids:
            yield function_ids, codes

    if remaining_skip != 0:
        raise RuntimeError(
            f"Resume offset for {dataset_name} exceeds its source row count."
        )


def process_model_dataset(
    spec: ModelSpec,
    dataset_name: str,
    tokenizer,
    model,
    revision: str,
    mode_token_id: Optional[int],
) -> dict[str, Any]:
    expected_rows = SOURCE_COUNTS[dataset_name]
    success_path = unit_success_path(spec.name, dataset_name)

    shard_metadata = {
        "model_name": spec.name,
        "model_id": spec.model_id,
        "model_revision": revision,
        "pooling": spec.pooling,
        "input_mode": spec.input_mode,
        "max_sequence_length": str(MAX_SEQUENCE_LENGTH),
        "window_overlap_tokens": str(WINDOW_OVERLAP_TOKENS),
        "window_aggregation": "newly_covered_token_weighted_mean",
        "l2_normalized": str(L2_NORMALIZE),
        "inference_dtype": INFERENCE_DTYPE,
        "output_dtype": OUTPUT_DTYPE,
        "self_pair_policy": "all_self_pairs_removed",
        "embedding_pipeline_version": EMBEDDING_PIPELINE_VERSION,
        "source_content_version": EXPECTED_SOURCE_CONTENT_VERSION,
        "source_selection": SELECTION_DESCRIPTIONS[dataset_name],
    }

    if unit_shards(spec.name, dataset_name) and not FORCE_REBUILD:
        assert_existing_shards_compatible(
            spec.name,
            dataset_name,
            shard_metadata,
        )

    existing_validation = validate_unit(
        spec.name,
        dataset_name,
        expected_rows,
        spec.expected_dimension,
    )
    if existing_validation.get("valid") and not FORCE_REBUILD:
        print(
            f"[SKIP] {spec.name} / {dataset_name}: validated output exists."
        )
        return existing_validation

    if FORCE_REBUILD:
        for path in unit_shards(spec.name, dataset_name):
            path.unlink(missing_ok=True)
        success_path.unlink(missing_ok=True)

    already_completed = completed_rows(spec.name, dataset_name)
    if already_completed > expected_rows:
        raise RuntimeError(
            f"{spec.name}/{dataset_name} has more output rows than source rows."
        )

    shard_index = next_shard_index(spec.name, dataset_name)
    buffered_ids: list[str] = []
    buffered_vectors: list[np.ndarray] = []

    processed_new = 0
    total_tokens = 0
    total_windows = 0

    progress = tqdm(
        total=expected_rows,
        initial=already_completed,
        desc=f"{spec.name} / {dataset_name}",
        unit="function",
    )

    def flush_rows(row_count: int) -> None:
        nonlocal buffered_ids, buffered_vectors, shard_index
        row_ids = buffered_ids[:row_count]
        row_vectors = np.vstack(buffered_vectors[:row_count])
        buffered_ids = buffered_ids[row_count:]
        buffered_vectors = buffered_vectors[row_count:]

        path = write_embedding_shard(
            spec.name,
            dataset_name,
            shard_index,
            row_ids,
            row_vectors,
            spec.expected_dimension,
            shard_metadata,
        )
        print(
            f"Wrote {path.name}: {len(row_ids):,} rows, "
            f"{path.stat().st_size / 2**20:.1f} MiB"
        )
        shard_index += 1

    for function_ids, codes in iter_source_batches(
        dataset_name,
        already_completed,
    ):
        vectors, token_counts, window_counts = embed_function_batch(
            codes,
            spec,
            tokenizer,
            model,
            mode_token_id,
            MODEL_WINDOW_BATCH_SIZES[spec.name],
        )

        buffered_ids.extend(function_ids)
        buffered_vectors.extend(list(vectors))
        processed_new += len(function_ids)
        total_tokens += sum(token_counts)
        total_windows += sum(window_counts)
        progress.update(len(function_ids))

        while len(buffered_ids) >= ROWS_PER_SHARD:
            flush_rows(ROWS_PER_SHARD)

    if buffered_ids:
        flush_rows(len(buffered_ids))

    progress.close()

    validation = validate_unit(
        spec.name,
        dataset_name,
        expected_rows,
        spec.expected_dimension,
    )
    report = {
        **validation,
        "model": asdict(spec),
        "model_revision": revision,
        "dataset": dataset_name,
        "new_rows": processed_new,
        "new_tokens": total_tokens,
        "new_windows": total_windows,
        "completed_at": datetime.now(timezone.utc).isoformat(),
    }
    success_path.write_text(
        json.dumps(report, indent=2, sort_keys=True),
        encoding="utf-8",
    )

    if not validation["valid"]:
        raise RuntimeError(
            f"Validation failed for {spec.name}/{dataset_name}: {validation}"
        )
    return report

## 14. GPTCloneBench embedding metadata and publication

The output Dataset contains four embedding units plus the exact clean-data pair,
code, and metadata files needed by the downstream classifier. A new private
Dataset is created on the first run and versioned on later runs.


In [ ]:
def validate_metadata_fields() -> None:
    if not OUTPUT_DATASET_TITLE.strip():
        raise ValueError("The output Dataset title is empty.")
    if not 20 <= len(OUTPUT_DATASET_SUBTITLE) <= 80:
        raise ValueError("Kaggle Dataset subtitle length must be between 20 and 80.")

def build_manifest() -> dict[str, Any]:
    units = []
    for model_name in MODELS_TO_RUN:
        spec = MODEL_SPECS[model_name]
        units.append({
            "model": model_name,
            "dataset": "GPTCloneBench",
            **validate_unit(model_name, "GPTCloneBench", SOURCE_COUNTS["GPTCloneBench"], spec.expected_dimension),
        })
    return {
        "format_version": "3.0.0",
        "embedding_pipeline_version": EMBEDDING_PIPELINE_VERSION,
        "source_content_version": EXPECTED_SOURCE_CONTENT_VERSION,
        "source_data_format": EXPECTED_SOURCE_DATA_FORMAT,
        "source_dataset_key": EXPECTED_SOURCE_DATASET_KEY,
        "source_dataset_root": str(SOURCE_ROOT),
        "source_input_method": SOURCE_INPUT_METHOD,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "output_dataset": OUTPUT_DATASET_HANDLE,
        "visibility": OUTPUT_DATASET_VISIBILITY,
        "private": True,
        "models": [asdict(MODEL_SPECS[name]) for name in MODELS_TO_RUN],
        "datasets": ["GPTCloneBench"],
        "source_function_counts": SOURCE_COUNTS,
        "full_source_function_counts": FULL_SOURCE_COUNTS,
        "version_3_pair_counts": EXPECTED_V3_PAIR_COUNTS,
        "version_3_positive_counts": EXPECTED_V3_POSITIVE_COUNTS,
        "self_pair_rows": 0,
        "bundled_classifier_inputs": ["pairs.csv", "codes.jsonl", "source_metadata.json"],
        "columns": {
            "function_id": "string", "dataset": "string",
            "embedding": f"fixed-size list<{OUTPUT_DTYPE}>[768]",
        },
        "pooling": {
            "CodeBERT": "first <s>/CLS hidden state",
            "GraphCodeBERT": "first <s>/CLS hidden state",
            "UniXCode": "official encoder-only masked mean",
            "CodeT5": "masked mean of encoder hidden states",
        },
        "graphcodebert": {"mode": "sequence-only", "explicit_data_flow_graph": False},
        "windowing": {
            "max_sequence_length": MAX_SEQUENCE_LENGTH,
            "overlap_content_tokens": WINDOW_OVERLAP_TOKENS,
            "aggregation": "newly-covered-token weighted mean",
        },
        "inference_dtype": INFERENCE_DTYPE,
        "output_dtype": OUTPUT_DTYPE,
        "l2_normalized": L2_NORMALIZE,
        "quick_test": QUICK_TEST,
        "units": units,
        "size_preflight": SIZE_PREFLIGHT,
        "total_output_bytes": output_files_size(),
    }

def create_cover_image() -> Path:
    path = OUTPUT_ROOT / "dataset-cover-image.png"
    image = Image.new("RGB", (1120, 560), (18, 24, 38))
    draw = ImageDraw.Draw(image)
    try:
        title_font = ImageFont.load_default(size=46)
        body_font = ImageFont.load_default(size=25)
        small_font = ImageFont.load_default(size=20)
    except TypeError:
        title_font = body_font = small_font = ImageFont.load_default()
    draw.text((70, 90), "GPTCloneBench V3 Clean Embeddings", fill=(244, 247, 252), font=title_font)
    draw.text((70, 185), "CodeBERT | GraphCodeBERT | UniXcoder | CodeT5", fill=(151, 203, 255), font=body_font)
    draw.text((70, 275), "5,924 functions | 5,924 pairs | no self-pairs", fill=(214, 221, 232), font=small_font)
    draw.text((70, 355), "Exact train/valid/test artifacts bundled for classification", fill=(214, 221, 232), font=small_font)
    image.save(path)
    return path

def write_output_metadata() -> dict[str, Any]:
    validate_metadata_fields()
    copy_source_artifacts()
    manifest = build_manifest()
    (OUTPUT_ROOT / "embedding_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    readme = f"""# GPTCloneBench V3 clean function embeddings

This private Dataset contains embeddings for all 5,924 functions in the no-self-pair GPTCloneBench V3 clean-data release.

Models: CodeBERT, GraphCodeBERT, UniXcoder, and CodeT5. Each Parquet row contains function_id, dataset, and a 768-dimensional embedding. The exact pairs.csv, codes.jsonl, and source_metadata.json files are bundled so the classification notebook needs only the saved embedding notebook output as input.

Split rows: train 4,144; valid 886; test 894. Every split is class-balanced and contains no self-pairs.
"""
    (OUTPUT_ROOT / "README.md").write_text(readme, encoding="utf-8")
    cover_path = create_cover_image()
    metadata = {
        "title": OUTPUT_DATASET_TITLE, "id": OUTPUT_DATASET_HANDLE,
        "licenses": [{"name": "other"}], "subtitle": OUTPUT_DATASET_SUBTITLE,
        "description": readme, "image": cover_path.name,
        "keywords": ["code clone detection", "code embeddings", "GPTCloneBench", "CodeBERT"],
    }
    (OUTPUT_ROOT / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return manifest


In [ ]:
def wait_for_dataset_status(handle: str) -> str:
    last_output = ""
    for attempt in range(1, 31):
        result = run_command(
            ["kaggle", "datasets", "status", handle],
            check=False,
            quiet=True,
            timeout=120,
        )
        last_output = combined_output(result)
        if result.returncode == 0:
            lowered = last_output.casefold()
            if not any(marker in lowered for marker in ("failed", "error")):
                return last_output
        if attempt < 30:
            time.sleep(10)
    raise RuntimeError(
        "The uploaded Dataset did not reach a readable status:\n" + last_output
    )


def publish_private_output(version_notes: str) -> dict[str, Any]:
    if QUICK_TEST:
        return {
            "status": "not_uploaded",
            "reason": "Quick-test output is never published.",
        }
    if not PUBLISH_TO_KAGGLE:
        return {"status": "disabled"}

    if OUTPUT_DATASET_VISIBILITY != "private":
        raise RuntimeError(
            "This notebook is configured to publish only a private Kaggle "
            "Dataset."
        )

    manifest = write_output_metadata()
    enforce_output_size_limit()

    completed_units = [
        unit for unit in manifest["units"] if unit.get("valid")
    ]
    invalid_nonempty_units = [
        unit
        for unit in manifest["units"]
        if unit.get("rows", 0) > 0 and not unit.get("valid")
    ]
    if invalid_nonempty_units:
        raise RuntimeError(
            "Output contains non-empty invalid units: "
            + json.dumps(invalid_nonempty_units, indent=2)
        )

    exists = owned_dataset_exists(OUTPUT_DATASET_HANDLE)
    if REQUIRE_EXISTING_OUTPUT_DATASET and not exists:
        raise RuntimeError(
            f"Dataset {OUTPUT_DATASET_HANDLE!r} no longer exists. "
            "Publication requires an existing output Dataset."
        )
    if exists and not ALLOW_VERSION_IF_EXISTS:
        raise RuntimeError(
            f"Dataset {OUTPUT_DATASET_HANDLE!r} already exists and "
            "versioning is disabled."
        )
    if exists:
        command = [
            "kaggle", "datasets", "version", "--path", str(OUTPUT_ROOT),
            "--message", version_notes, "--keep-tabular", "--dir-mode", "skip",
        ]
        action = "version_private_existing"
    else:
        command = [
            "kaggle", "datasets", "create", "--path", str(OUTPUT_ROOT),
            "--keep-tabular", "--dir-mode", "skip",
        ]
        action = "create_private_new"

    run_command(command, timeout=14_400)
    remote_status = wait_for_dataset_status(OUTPUT_DATASET_HANDLE)

    return {
        "status": "uploaded",
        "action": action,
        "handle": OUTPUT_DATASET_HANDLE,
        "visibility": OUTPUT_DATASET_VISIBILITY,
        "private": OUTPUT_DATASET_VISIBILITY == "private",
        "completed_units": len(completed_units),
        "output_gib": output_files_size() / 2**30,
        "remote_status": remote_status,
        "url": "https://www.kaggle.com/datasets/" + OUTPUT_DATASET_HANDLE,
    }

## 15. Run the GPTCloneBench embedding pipeline

Each of the four checkpoints processes the same validated set of 5,924 code
endpoints. Publication is deferred until every embedding unit passes validation.


In [ ]:
validate_metadata_fields()
write_output_metadata()

RUN_REPORTS: list[dict[str, Any]] = []
SMOKE_TEST_REPORTS: list[dict[str, Any]] = []
PUBLICATION_REPORTS: list[dict[str, Any]] = []

for model_name in MODELS_TO_RUN:
    spec = MODEL_SPECS[model_name]

    print("\n" + "=" * 90)
    print("MODEL:", model_name)
    print("Checkpoint:", spec.model_id)
    print("Pooling:", spec.pooling)
    print("Input mode:", spec.input_mode)
    print("=" * 90)

    tokenizer, model, revision, mode_token_id = load_embedding_model(spec)
    print("Resolved model revision:", revision)

    smoke_report = smoke_test_model(
        spec,
        tokenizer,
        model,
        mode_token_id,
    )
    smoke_report["revision"] = revision
    SMOKE_TEST_REPORTS.append(smoke_report)

    for dataset_name in DATASETS_TO_RUN:
        print("\n" + "-" * 90)
        print(f"{model_name} / {dataset_name}")
        print("-" * 90)

        report = process_model_dataset(
            spec,
            dataset_name,
            tokenizer,
            model,
            revision,
            mode_token_id,
        )
        RUN_REPORTS.append(report)
        print(json.dumps(report, indent=2, default=str))

    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    write_output_metadata()

    if PUBLISH_AFTER_EACH_MODEL:
        publication = publish_private_output(
            OUTPUT_VERSION_NOTES + f"; completed {model_name} at "
            + datetime.now(timezone.utc).isoformat()
        )
        PUBLICATION_REPORTS.append(publication)
        print(json.dumps(publication, indent=2))

if PUBLISH_TO_KAGGLE and not PUBLISH_AFTER_EACH_MODEL:
    print(
        "Publication is deferred until final validation so the output "
        "Dataset version is published exactly once."
    )

## 16. Final validation and publication

The final cell requires one finite, non-zero vector per GPTCloneBench code ID
for every configured model before publishing the private output Dataset.


In [ ]:
FINAL_MANIFEST = write_output_metadata()
FINAL_VALIDATION = {
    "valid": all(
        unit.get("valid", False)
        for unit in FINAL_MANIFEST["units"]
    ),
    "units": FINAL_MANIFEST["units"],
    "source_function_counts": SOURCE_COUNTS,
    "full_source_function_counts": FULL_SOURCE_COUNTS,
    "source_content_version": EXPECTED_SOURCE_CONTENT_VERSION,
    "embedding_pipeline_version": EMBEDDING_PIPELINE_VERSION,
    "version_3_pair_counts": EXPECTED_V3_PAIR_COUNTS,
    "total_selected_functions_per_model": sum(SOURCE_COUNTS.values()),
    "models": MODELS_TO_RUN,
    "output_gib": FINAL_MANIFEST["total_output_bytes"] / 2**30,
    "dataset_handle": None,
    "publication_mode": "saved_notebook_output",
    "output_directory": str(OUTPUT_ROOT),
    "visibility": OUTPUT_DATASET_VISIBILITY,
    "private": OUTPUT_DATASET_VISIBILITY == "private",
    "smoke_tests": SMOKE_TEST_REPORTS,
    "publication_reports": PUBLICATION_REPORTS,
}

(OUTPUT_ROOT / "final_validation.json").write_text(
    json.dumps(FINAL_VALIDATION, indent=2, sort_keys=True),
    encoding="utf-8",
)

print(json.dumps(FINAL_VALIDATION, indent=2, default=str))

if not FINAL_VALIDATION["valid"]:
    raise RuntimeError(
        "At least one configured model–dataset unit is incomplete or invalid."
    )

if PUBLISH_TO_KAGGLE:
    final_publication = publish_private_output(
        OUTPUT_VERSION_NOTES
        + "; final validation completed at "
        + datetime.now(timezone.utc).isoformat()
    )
    print(json.dumps(final_publication, indent=2))

print("\nAll configured embeddings are complete and valid.")
print("Saved notebook output directory:", OUTPUT_ROOT)
print("Use Save Version, then attach this Notebook Output to the classification notebook.")